## SimpleITK/nilearn may require py39 environment

In [ ]:
import os
import pandas as pd
import SimpleITK as sitk
import numpy as np
from nilearn import plotting
import matplotlib.pyplot as plt
import seaborn as sns

PATH = "/home/kibr/Work/Vascular_atlas/"
os.chdir(PATH)

In [ ]:
## Data required for plotting
atlas_csv_path="./Data/Ding_Atlas_Table.csv"
atlas_nii_path="./Data/annotation_full.nii"

vascular_meta_path = "./Data/2_Vascular_Mapped_Region_ID.csv"
gene_expr_mtx_path = "./Results/Revision_2/combined_expression_density.csv"


In [ ]:
## Read in the data
Ding_Atlas_Table = pd.read_csv(atlas_csv_path)
volume = sitk.ReadImage(atlas_nii_path)

Vascular_meta = pd.read_csv(vascular_meta_path)
gene_expr_mtx = pd.read_csv(gene_expr_mtx_path,index_col=0)

In [ ]:
## transpose the gene expression matrix so that rows are regions and columns are genes
gene_expr_mtx = gene_expr_mtx.T

gene_expr_mtx.tail(15)

In [ ]:
## Select specific column as the index for mapping purpose
Ding_Atlas_Table.index = Ding_Atlas_Table["Unique Values/Labels"].astype(str)
# Vascular_meta.index = Vascular_meta["regionID"].astype(str)+"_"+Vascular_meta["Final_abb"]
Vascular_meta.index = Vascular_meta["region_name"].astype(str)
## Double checking if can be mapped
set(gene_expr_mtx.columns) == set(Vascular_meta.index)
## if false, check which ones are missing
set(gene_expr_mtx.columns) - set(Vascular_meta.index)

In [ ]:
def get_region_abundance_dict(gene_name, mtx, metadata_indexed, ding_atlas_reference):
    """
    Map expression values of a gene to Ding Atlas regions by averaging across regions
    that map to the same Ding structure.

    Parameters:
    - gene_name: gene of interest (row name in mtx)
    - mtx: DataFrame with gene expression, rows=genes, cols=region IDs (e.g., 1_HIP, 2_AMY)
    - metadata_indexed: DataFrame with region metadata, indexed by region IDs, includes 'Ding_Atlas_ID'
    - ding_atlas_reference: DataFrame indexed by Ding Atlas ID, includes 'Region Names'

    Returns:
    - dict of {ding_structure_name: averaged_expression_value}
    """
    gene_expr_series = mtx.loc[gene_name]
    
    expr_sum = {}
    count = {}

    for region_id, expr_value in gene_expr_series.items():
        if pd.isnull(expr_value):
            continue

        if region_id in metadata_indexed.index:
            atlas_ids_str = metadata_indexed.loc[region_id, "Ding_Atlas_ID"]
            if isinstance(atlas_ids_str, str):
                atlas_ids = [id_.strip() for id_ in atlas_ids_str.split(',')]
                for atlas_id in atlas_ids:
                    if atlas_id in ding_atlas_reference.index:
                        region_name = ding_atlas_reference.loc[atlas_id, "Region Names"]
                        if pd.notnull(region_name):
                            expr_sum[region_name] = expr_sum.get(region_name, 0) + float(expr_value)
                            count[region_name] = count.get(region_name, 0) + 1
                    else:
                        print(f"Atlas ID {atlas_id} not found in ding_atlas_reference")
            else:
                print(f"Invalid or missing Ding_Atlas_ID for region {region_id}")
        else:
            print(f"Region ID {region_id} not found in metadata_indexed")

    # Compute average expression for each region
    region_abundance_dict = {
        region: expr_sum[region] / count[region]
        for region in expr_sum
    }

    return region_abundance_dict

In [ ]:
def visualize_gene_abundance(region_abundance_dict,
                              atlas_csv_path="./Data/Ding_Atlas_Table.csv",
                              atlas_nii_path="./Data/annotation_full.nii",
                              output_nii_path="gene_abundance_map.nii",
                              cmap="Reds"):
    """
    Visualize gene abundance across regions using the Ding Atlas.
    
    Parameters:
    - region_abundance_dict: dict of {region_name: abundance_value}
    - atlas_csv_path: path to Ding_Atlas_Table.csv
    - atlas_nii_path: path to Ding annotation_full.nii file
    - output_nii_path: output filename for modified NIfTI
    - cmap: matplotlib colormap name
    """

    # Load atlas table and base annotation volume
    Ding_Atlas_Table = pd.read_csv(atlas_csv_path)
    volume = sitk.ReadImage(atlas_nii_path)
    arr = sitk.GetArrayFromImage(volume)
    arr_new = np.zeros_like(arr, dtype=np.float32)

    # Check for non-empty input
    if not region_abundance_dict:
        raise ValueError("region_abundance_dict is empty. Please provide region: abundance values.")

    # Get max value for normalization
    max_val = max(region_abundance_dict.values())
    if max_val == 0:
        raise ValueError("Maximum abundance value is 0. Cannot normalize.")

    # Fill new array with abundance values
    for region_name, gene_value in region_abundance_dict.items():
        match = Ding_Atlas_Table.loc[Ding_Atlas_Table['Region Names'] == region_name, "Unique Values/Labels"]
        if not match.empty:
            label_value = int(match.values[0])
            arr_new[arr == label_value] = gene_value
        else:
            print(f"Warning: Region '{region_name}' not found in Ding Atlas.")

    # Normalize to [-1, 1] to avoid auto-symmetric color scaling
    abs_max = max(abs(min(region_abundance_dict.values())), abs(max_val))
    arr_new /= abs_max

    # Save modified image
    modified_volume = sitk.GetImageFromArray(arr_new)
    modified_volume.CopyInformation(volume)
    sitk.WriteImage(modified_volume, output_nii_path)

    # Display interactive plot
    interactive_plot = plotting.view_img(output_nii_path, cmap=cmap,
                                         draw_cross=False, colorbar=True, threshold=0)

    return interactive_plot

In [ ]:
gene_name = "SLC16A1"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="viridis")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="viridis")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

gene_name = "TFRC"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="viridis")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="viridis")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

gene_name = "SLC2A1"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="viridis")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="viridis")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

In [ ]:
gene_name = "avg_astrocyte_density"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="vlag")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="vlag")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

gene_name = "avg_neuronal_density"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="vlag")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="vlag")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

gene_name = "SYT1"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="vlag")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="vlag")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")

gene_name = "NDRG2"
gene_map = get_region_abundance_dict(gene_name, gene_expr_mtx, Vascular_meta, Ding_Atlas_Table)
# gene_map

# Now pass it to your function
output_nii = f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii"
visualize_gene_abundance(gene_map, output_nii_path=output_nii,cmap="vlag")

interactive_plot = plotting.view_img(f"./Results/Revision_2/brain_vis/{gene_name}_expression.nii", cmap="vlag")
interactive_plot.save_as_html(f"./Results/Revision_2/brain_vis/{gene_name}_abundance_viewer.html")